# 3.9: A baseline recipe

| Ingredient | Default choice | Section |
|---|---|---|
| Initialization | He for ReLU-family, Xavier for tanh; small init for residual branch outputs | 3, 4 |
| Architecture | Residual blocks | 4 |
| Normalization | Pre-norm LayerNorm or RMSNorm (BatchNorm for conv nets with large batches) | 5 |
| Optimizer | AdamW, $\beta_1 = 0.9$, $\beta_2 \in [0.95, 0.999]$, weight decay about 0.01–0.1 on weight matrices | 6, 8 |
| Learning rate | Chosen by a range test or a short sweep | 7 |
| Schedule | Linear warmup, then cosine or linear decay | 7 |
| Gradient clipping | Global norm, threshold around 1.0 | 2 |
| Regularization | Dropout and augmentation as needed, judged from the train/validation gap | 8 |
| Precision | BF16 mixed precision where supported | This section |

Here is the recipe as a compact PyTorch training loop for a residual MLP classifier. It is deliberately minimal but contains every ingredient:


In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F

class Block(nn.Module):
    def __init__(self, d, p_drop=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fc1, self.fc2 = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.drop = nn.Dropout(p_drop)
    def forward(self, x):                       # pre-norm residual block
        return x + self.drop(self.fc2(F.gelu(self.fc1(self.norm(x)))))

class ResMLP(nn.Module):
    def __init__(self, d_in, d, n_blocks, n_classes):
        super().__init__()
        self.inp = nn.Linear(d_in, d)
        self.blocks = nn.ModuleList(Block(d) for _ in range(n_blocks))
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_classes)
        for b in self.blocks:                   # small init for residual outputs
            nn.init.normal_(b.fc2.weight, std=0.02 / math.sqrt(2 * n_blocks))
            nn.init.zeros_(b.fc2.bias)
    def forward(self, x):
        x = self.inp(x.flatten(1))
        for b in self.blocks:
            x = b(x)
        return self.head(self.norm(x))

def train(model, train_loader, total_steps, peak_lr=1e-3, warmup=500, device="cuda"):
    model.to(device).train()
    decay = [p for n, p in model.named_parameters() if p.ndim >= 2]
    no_decay = [p for n, p in model.named_parameters() if p.ndim < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.05},
                             {"params": no_decay, "weight_decay": 0.0}], lr=peak_lr)
    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        s = min(1.0, (step - warmup) / max(1, total_steps - warmup))
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * s))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    step = 0
    while step < total_steps:
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                loss = F.cross_entropy(model(x), y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); step += 1
            if step % 100 == 0:
                print(f"step {step} loss {loss.item():.4f} gnorm {gnorm:.2f} "
                      f"lr {sched.get_last_lr()[0]:.2e}")
            if step >= total_steps:
                break
